## 1. Network Structure Comparison

- Graph metrics: Degree distribution, clustering coefficient, diameter, betweenness centrality, modularity, etc.

- Community structure: How tightly knit are the communities (e.g., Hogwarts Houses vs. the Fellowship)?

- Central characters: Compare who acts as social hubs or bridges (e.g., Dumbledore vs. Gandalf).

- Resilience: What happens to the network if you remove key characters (simulate “killing off” Dumbledore or Frodo)?


### Question:
- *Which universe has a more centralized social structure?*
- *Is one universe more hierarchical or egalitarian in terms of interactions?*

## 2. Role and Archetype Similarity

- Use the graph structure to explore role equivalence:

- Identify structural equivalents across universes: e.g., Frodo ↔ Harry (protagonists), Gandalf ↔ Dumbledore (mentor), Sauron ↔ Voldemort (villain).

- Use graph embedding or role similarity algorithms (like node2vec) to find characters that occupy similar positions in their networks.

### Questions:

- *Can we identify parallel roles purely from network structure, without narrative context?*

## 3. Sentiment & Relationship Types

- Label edges as positive (friendship) or negative (conflict).

- Compare the balance of positive vs. negative relationships.

- See how negativity propagates (e.g., Voldemort and Sauron as sources of “negative influence”).

### Question:

- *Which universe is “happier” in its social graph?*
- *Is the Lord of the Rings network more polarized than the Harry Potter one?*

Spiseseddel/To-do list:
Harrypotter side:       https://harrypotter.fandom.com/wiki/Main_Page
Lord of the Rings side: https://lotr.fandom.com/wiki/Main_Page

1. Identificer karakter sider:
    - HP: https://harrypotter.fandom.com/wiki/Category:Individuals
    - LOTR: https://lotr.fandom.com/wiki/Category:Characters

2. Find ud af hvordan fandom.com api fungere (https://chatgpt.com/s/t_690b2537f16481919516ba98b1162d55)
3. Lav function som henter alt text fra en fandom.com side.
    1. Gå igennem alle links på category siden.
    2. Hvis link er kategori så åbn den nye kategori side og gå tilbage til 1, 
    hvis ikke så kig om den side er blevet downloadet, hvis ikke så download, ellers fortsæt.
    3. Gem både karakter og side tekst
4. Start analyse. Lav video på 1:30 min til projA 

In [1]:
import urllib.request, json, time, pickle

def get_category_members(url, page_name):
    base_url = f'https://{url}/api.php?'
    query = (
        f"{base_url}"
        f"action=query&list=categorymembers&cmtitle={urllib.parse.quote(page_name)}"
        f"&cmlimit=max&format=json"
    )
    request = urllib.request.Request(query)
    request.add_header('User-Agent', 'YourAppName/1.0 (s204229@student.dtu.dk)')
    with urllib.request.urlopen(request) as response:
        wikidata = json.loads(response.read())
        return [member['title'] for member in wikidata['query']['categorymembers']]


def extract_characters_and_categories(response, downloaded_categories):
    # Words that indicate non-character categories
    blacklist = ['places', 'creatures', 'locations', 'objects', 'languages',
                 'rivers', 'mountains', 'events', 'battles', 'wars', 'books',
                 'families', 'songs', 'terms', 'legends', 'real-world', 'media', 
                 'non-canonical', 'images', 'ranks', 'titles']
    
    characters = []
    categories = []

    for title in response:
        if title.startswith('Category:'):
            # Normalize for comparison
            lower = title.lower()
            if any(bad in lower for bad in blacklist):
                continue  # skip non-character categories
            if title not in downloaded_categories:
                categories.append(title)
        else:
            # Keep only likely character pages (heuristic: no colon, not "Category:", not image)
            if ':' not in title and not title.lower().startswith('list of'):
                characters.append(title)
    return list(set(characters)), list(set(categories))


def get_all_characters(url, root_category='Category:Characters'):
    downloaded_categories = [root_category]
    response = get_category_members(url, root_category)
    characters, categories = extract_characters_and_categories(response, downloaded_categories)

    while categories:
        current = categories.pop(0)
        downloaded_categories.append(current)
        #print(f"{len(categories)} remaining. Fetching: {current}")
        response = get_category_members(url, current)
        char, cat = extract_characters_and_categories(response, downloaded_categories)
        characters = list(set(characters + char))
        categories = list(set(categories))# + cat))
        time.sleep(0.5)

    return sorted(set(characters))

def download_pages(url, page_names, batch_size=25):
    api_url = f'https://{url}/api.php'
    results = []

    for i in range(0, len(page_names), batch_size):
        batch = page_names[i:i+batch_size]
        clean_batch = [t for t in batch if t and not any(c in t for c in '\n\r{}')]
        if not clean_batch:
            continue

        data = {
            'action': 'query',
            'prop': 'revisions',
            'rvprop': 'content',
            'titles': '|'.join(clean_batch),
            'format': 'json'
        }

        encoded = urllib.parse.urlencode(data).encode('utf-8')
        request = urllib.request.Request(api_url, data=encoded)
        request.add_header('User-Agent', 'YourAppName/1.0 (s204229@student.dtu.dk)')

        try:
            with urllib.request.urlopen(request) as response:
                wikidata = json.loads(response.read())
                for key, page in wikidata['query']['pages'].items():
                    revisions = page.get('revisions', [{}])
                    content = revisions[0].get('*', '') if revisions else ''
                    results.append({
                        'page_name': page.get('title', ''),
                        'content': content
                    })
        except urllib.error.HTTPError as e:
            print(f"HTTP Error {e.code} on batch {i//batch_size + 1}: {e.reason}")
        time.sleep(0.3)
    return results


In [2]:
url             = 'lotr.fandom.com'
category_name   = 'Category:Characters'
page_names      = get_all_characters(url, category_name)
print(f'There are {len(page_names)} characters in Lord of the Rings')
print(f'First 10 pages:\n {page_names[10:]}')

lotr_pages = download_pages(url, page_names)
with open("lotr_characters.pkl", "wb") as f:   # 'wb' = write binary
    pickle.dump(lotr_pages, f)

There are 895 characters in Lord of the Rings
First 10 pages:
 ['Agathor', 'Aghan', 'Aglahad', 'Ailinel', 'Ainur', 'Alatar', 'Albert Dreary', 'Aldamir', 'Aldor (film character)', 'Algund', 'Alinel', 'Almarian', 'Almiel', 'Almáriel', 'Alphros', 'Amandil', 'Amdír', 'Amlach', 'Amlaith', 'Ammred', 'Amnon', 'Amras', 'Amrod', 'Amroth', 'Amrothos', 'Amárië', 'Anairë', 'Anardil', 'Ancalagon', 'Andreth', 'Andróg', 'Andvír', 'Angamaitë', 'Angelimir', 'Angrim', 'Angrod', 'Annael', 'Anárion', 'Ar-Adûnakhôr', 'Ar-Gimilzôr', 'Ar-Pharazôn', 'Ar-Sakalthôr', 'Ar-Zimrathôn', 'Arachon', 'Arador', 'Araglas', 'Aragorn I', 'Aragorn II', 'Aragost', 'Arahad I', 'Arahad II', 'Arahael', 'Aranarth', 'Arantar', 'Aranuir', 'Aranwë', 'Araphant', 'Araphor', 'Arassuil', 'Aratan', 'Aratar', 'Arathorn I', 'Arathorn II', 'Araval', 'Aravir', 'Aravorn', 'Arciryas', 'Ardamir', 'Aredhel', 'Argeleb I', 'Argeleb II', 'Argon', 'Argonui', 'Arien', 'Arminas', 'Arondir', 'Artamir', 'Arthad', 'Arvedui', 'Arvegil', 'Arveleg I', 'Ar

In [3]:
url             = 'harrypotter.fandom.com'
category_name   = 'Category:Individuals'
page_names      = get_all_characters(url, category_name)
print(f'There are {len(page_names)} characters in Harry Potter')
print(f'First 10 pages:\n {page_names[10:]}')

hp_pages = download_pages(url, page_names)
with open("hp_characters.pkl", "wb") as f:   # 'wb' = write binary
    pickle.dump(hp_pages, f)

There are 998 characters in Harry Potter
First 10 pages:
 ['Adda', 'Addison Fawley', 'Adelaide Oakes', "Adelaide Oakes's father", "Adelaide Oakes's mother", 'Adrian Pucey', "Aesop Sharp's partner", 'African prince', 'Aged witch', 'Agnes Nutt', 'Aidan Sprottle', 'Ailsa Travers', 'Aisha', 'Alasdair Maddock', 'Alastor Moody', 'Albert Runcorn', 'Albert Stump', 'Albie Weekes', 'Albus Dumbledore', "Albus Dumbledore's great-great-grandfather", 'Alexandra Ricketts', 'Alice Fitzwarren', "Alice Longbottom's midwife", 'Alison Denbright', 'Altheda', 'Amata', "Amata's lover", "Amelia Bones's assistant", 'Amir', 'Amos Diggory', 'Amos Hopkins', 'Amrose Swott', 'Anabel Calhoun', 'Ananya Gopal', 'Ancient African sorcerer', 'Andrew Larson', 'Andromeda Tonks', 'Angel', 'Angel of Death', 'Angry wizard', 'Angus Buchanan', 'Angus Sweeting', 'Annalena Gleam', "Antioch Peverell's enemy", "Antioch Peverell's killer", 'Apothecary shopkeeper', 'Aquila Greengrass', 'Architect of Hogwarts', 'Armand Malfoy', 'Armen